# 2. Extracting Confidence Regions from the Log-Likelihood

Building on [01_maximum_likelihood_fitting.ipynb](01_maximum_likelihood_fitting.ipynb), we derive **68 % and 95 % confidence intervals / regions** for the parameters of the Gaussian + exponential mixture using the **profile likelihood** and **Wilks' theorem**.

We consider the same PDF

$$f(x;\theta,\mu,\sigma,\lambda) = \theta \cdot \mathcal{N}(x;\mu,\sigma) + (1-\theta) \cdot \mathcal{E}(x;\lambda)$$

where:
- $\theta \in [0,1]$ is the **parameter of interest** (signal fraction)
- $\mu,\sigma$ are the mean and width of the Gaussian (nuisance parameters)
- $\lambda > 0$ is the rate of the exponential (nuisance parameter)

We will:
1. Reproduce the data generation and MLE from notebook 01  
2. Review the profile-likelihood and Wilks' theorem  
3. Look at one-dimensional confidence intervals for each parameter  
4. Study two-dimensional confidence regions for all pairs of parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, chi2
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)

## 1. Data Generation and MLE (recap from notebook 01)

We reuse **exactly the same dataset** as in
[01_maximum_likelihood_fitting.ipynb](01_maximum_likelihood_fitting.ipynb)
(identical random seed → identical data). The data visualisation is shown there;
here we only reproduce the code needed to obtain the MLE and the
internal-parameter vector used by the profiling routines below.

In [ ]:
# ------- true parameter values -------
theta_true  = 0.3    # signal fraction
mu_true     = 5.0    # Gaussian mean
sigma_true  = 0.5    # Gaussian width
lambda_true = 0.5    # exponential rate  (mean = 1/lambda)

n_samples = 2000

# ------- sampling from the mixture -------
n_signal     = rng.binomial(n_samples, theta_true)
n_background = n_samples - n_signal

x_signal     = rng.normal(mu_true, sigma_true, size=n_signal)
x_background = rng.exponential(1.0 / lambda_true, size=n_background)

x_data = np.concatenate([x_signal, x_background])
rng.shuffle(x_data)

print(f"Generated {len(x_data)} events  "
      f"({n_signal} signal + {n_background} background)")

In [ ]:
# Same NLL function as in notebook 01 (reproduced here for self-containedness).
def nll_full(params, data):
    """Negative log-likelihood with all four parameters free.
    Internal parameterisation: (theta, mu, log_sigma, log_lam).
    """
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma)
    lam   = np.exp(log_lam)

    if not (0.0 < theta < 1.0):
        return np.inf

    log_sig_term = np.log(theta)       + norm.logpdf(data, mu, sigma)
    log_bkg_term = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    return -np.sum(np.logaddexp(log_sig_term, log_bkg_term))

In [ ]:
theta0, mu0, sigma0, lam0 = 0.2, x_data.mean(), 0.6, 0.6
x0 = [theta0, mu0, np.log(sigma0), np.log(lam0)]

result = minimize(
    nll_full, x0=x0, args=(x_data,),
    method='Nelder-Mead',
    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000}
)

theta_hat, mu_hat, log_sigma_hat, log_lam_hat = result.x
sigma_hat = np.exp(log_sigma_hat)
lam_hat   = np.exp(log_lam_hat)
nll_min   = result.fun

# Collect MLE in internal space for later re-use
mle_internal = np.array([theta_hat, mu_hat, log_sigma_hat, log_lam_hat])

print("MLE (all parameters free)")
print(f"  theta  = {theta_hat:.4f}  (true: {theta_true})")
print(f"  mu     = {mu_hat:.4f}   (true: {mu_true})")
print(f"  sigma  = {sigma_hat:.4f}  (true: {sigma_true})")
print(f"  lambda = {lam_hat:.4f}  (true: {lambda_true})")
print(f"  NLL = {nll_min:.4f},  success: {result.success}")

## 2. Profile Likelihood and Wilks' Theorem

### Profile likelihood

The **profile NLL** for a subset of parameters $\psi$ (the *parameters of interest*)
is obtained by minimising the NLL over all remaining *nuisance* parameters $\nu$:

$$\text{NLL}_{\text{prof}}(\psi) \;=\; \min_{\nu}\; \text{NLL}(\psi,\nu)$$

### Wilks' theorem

Under regularity conditions, twice the difference in profile NLL follows asymptotically a chi-squared
distribution:

$$2\,\Delta\text{NLL}(\psi) \;=\; 2\bigl[\text{NLL}_{\text{prof}}(\psi)
- \text{NLL}_{\text{prof}}(\hat{\psi})\bigr] \;\xrightarrow{n\to\infty}\; \chi^2_k$$

where $k$ is the number of parameters of interest.

| Parameters of interest | 68.3 % threshold | 95.0 % threshold |
|---|---|---|
| $k=1$ (1-D intervals) | $\Delta\text{NLL} = 0.50$ | $\Delta\text{NLL} = 1.92$ |
| $k=2$ (2-D regions)   | $\Delta\text{NLL} = 1.15$ | $\Delta\text{NLL} = 3.00$ |

In [ ]:
# Wilks' theorem thresholds
cl68_1d = chi2.ppf(0.6827, df=1) / 2   # approx 0.500
cl95_1d = chi2.ppf(0.9500, df=1) / 2   # approx 1.921

cl68_2d = chi2.ppf(0.6827, df=2) / 2   # approx 1.148
cl95_2d = chi2.ppf(0.9500, df=2) / 2   # approx 2.996

print(f"1-D thresholds:  68.3 % -> DNLL = {cl68_1d:.3f},  "
      f"95.0 % -> DNLL = {cl95_1d:.3f}")
print(f"2-D thresholds:  68.3 % -> DNLL = {cl68_2d:.3f},  "
      f"95.0 % -> DNLL = {cl95_2d:.3f}")

## 3. One-Dimensional Confidence Intervals

For each parameter in turn, we fix that parameter to a grid of values and minimise
the NLL over the other three to obtain the profile NLL.

In [ ]:
def profile_nll_1d(fixed_idx, fixed_val_internal, data, mle_internal):
    """1-D profile NLL.

    Parameters
    ----------
    fixed_idx : int
        Index of the parameter to fix in the internal space
        (0 = theta, 1 = mu, 2 = log_sigma, 3 = log_lam).
    fixed_val_internal : float
        Value of the fixed parameter in internal space.
    data : array
    mle_internal : array
        MLE values [theta, mu, log_sigma, log_lam].
    """
    free_idx = [i for i in range(4) if i != fixed_idx]
    x0 = mle_internal[free_idx]

    def obj(free_vals):
        params = mle_internal.copy()
        params[free_idx]  = free_vals
        params[fixed_idx] = fixed_val_internal
        theta, mu, log_s, log_l = params
        if not (0.0 < theta < 1.0):
            return 1e10
        sigma, lam = np.exp(log_s), np.exp(log_l)
        ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
        lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
        return -np.sum(np.logaddexp(ls, lb))

    res = minimize(obj, x0=x0, method='Nelder-Mead',
                   options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 10000})
    return res.fun

In [ ]:
# Parameter grids in *display* space (sigma and lambda are positive)
grids = {
    'theta':  np.linspace(0.12, 0.55, 60),
    'mu':     np.linspace(4.60, 5.40, 60),
    'sigma':  np.linspace(0.28, 0.80, 60),
    'lambda': np.linspace(0.28, 0.80, 60),
}

# Mapping: display name -> (internal index, transform to internal space)
param_info = {
    'theta':  (0, lambda v: v),
    'mu':     (1, lambda v: v),
    'sigma':  (2, np.log),      # internal = log(sigma)
    'lambda': (3, np.log),      # internal = log(lambda)
}

# MLE in display space (for reference lines)
mle_display = {
    'theta':  theta_hat,
    'mu':     mu_hat,
    'sigma':  sigma_hat,
    'lambda': lam_hat,
}
true_display = {
    'theta':  theta_true,
    'mu':     mu_true,
    'sigma':  sigma_true,
    'lambda': lambda_true,
}

# Compute profile NLL on each grid
delta_nll_1d = {}
for name, grid in grids.items():
    idx, to_internal = param_info[name]
    vals = np.array([
        profile_nll_1d(idx, to_internal(v), x_data, mle_internal)
        for v in grid
    ])
    delta_nll_1d[name] = vals - vals.min()
    print(f"  {name}: done")

In [ ]:
labels = {
    'theta':  r'$\theta$',
    'mu':     r'$\mu$',
    'sigma':  r'$\sigma$',
    'lambda': r'$\lambda$',
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for ax, (name, grid) in zip(axes, grids.items()):
    dnll = delta_nll_1d[name]
    ax.plot(grid, dnll, 'b-', lw=2, label=r'$\Delta$NLL')
    ax.axhline(cl68_1d, color='orange', ls='--',
               label=f'68 % ($\\Delta$NLL = {cl68_1d:.2f})')
    ax.axhline(cl95_1d, color='red',    ls='--',
               label=f'95 % ($\\Delta$NLL = {cl95_1d:.2f})')
    ax.axvline(mle_display[name], color='green', ls=':', lw=1.5,
               label=f'MLE = {mle_display[name]:.3f}')
    ax.axvline(true_display[name], color='black', ls=':', lw=1.5,
               label=f'True = {true_display[name]:.3f}')
    ax.set_xlabel(labels[name], fontsize=13)
    ax.set_ylabel(r'$\Delta$NLL', fontsize=12)
    ax.set_title(f'Profile likelihood \u2013 {labels[name]}', fontsize=12)
    ax.set_ylim(-0.1, 6.5)
    ax.legend(fontsize=8)

plt.suptitle(r'1-D Profile $\Delta$NLL and Confidence Intervals', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("One-dimensional confidence intervals (Wilks' theorem)\n")
print(f"{'Parameter':<10}  {'68% CI':^24}  {'95% CI':^24}")
print("-" * 64)
for name, grid in grids.items():
    dnll = delta_nll_1d[name]
    in68 = grid[dnll <= cl68_1d]
    in95 = grid[dnll <= cl95_1d]
    ci68 = f"[{in68.min():.4f}, {in68.max():.4f}]" if len(in68) else "-"
    ci95 = f"[{in95.min():.4f}, {in95.max():.4f}]" if len(in95) else "-"
    print(f"{labels[name]:<10}  {ci68:^24}  {ci95:^24}")

## 4. Two-Dimensional Confidence Regions

For each pair of parameters we fix both to a grid of values and minimise the NLL
over the remaining two.  The 68 % and 95 % contours are drawn at
$\Delta\text{NLL} = 1.15$ and $\Delta\text{NLL} = 3.00$ respectively.

In [ ]:
def profile_nll_2d(idx1, val1_internal, idx2, val2_internal, data, mle_internal):
    """2-D profile NLL: fix two parameters, optimise over the other two."""
    fixed = {idx1: val1_internal, idx2: val2_internal}
    free_idx = [i for i in range(4) if i not in fixed]
    x0 = mle_internal[free_idx]

    def obj(free_vals):
        params = mle_internal.copy()
        params[free_idx] = free_vals
        for fi, fv in fixed.items():
            params[fi] = fv
        theta, mu, log_s, log_l = params
        if not (0.0 < theta < 1.0):
            return 1e10
        sigma, lam = np.exp(log_s), np.exp(log_l)
        ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
        lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
        return -np.sum(np.logaddexp(ls, lb))

    if len(free_idx) == 0:
        return obj([])
    res = minimize(obj, x0=x0, method='Nelder-Mead',
                   options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 10000})
    return res.fun

In [ ]:
# Coarser grids for the 2-D scan (each point requires a 2-D minimisation)
grids_2d = {
    'theta':  np.linspace(0.14, 0.52, 22),
    'mu':     np.linspace(4.65, 5.35, 22),
    'sigma':  np.linspace(0.30, 0.76, 22),
    'lambda': np.linspace(0.30, 0.76, 22),
}

pairs = [
    ('theta', 'mu'),
    ('theta', 'sigma'),
    ('theta', 'lambda'),
    ('mu',    'sigma'),
    ('mu',    'lambda'),
    ('sigma', 'lambda'),
]

delta_nll_2d = {}
for (n1, n2) in pairs:
    g1 = grids_2d[n1]
    g2 = grids_2d[n2]
    idx1, to1 = param_info[n1]
    idx2, to2 = param_info[n2]
    Z = np.zeros((len(g2), len(g1)))
    for j, v2 in enumerate(g2):
        for i, v1 in enumerate(g1):
            Z[j, i] = profile_nll_2d(
                idx1, to1(v1), idx2, to2(v2), x_data, mle_internal
            )
    Z -= Z.min()
    delta_nll_2d[(n1, n2)] = Z
    print(f"  ({n1}, {n2}): done")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

contour_levels = [cl68_2d, cl95_2d]
contour_colors = ['orange', 'red']
contour_labels = [f'68 % ($\\Delta$NLL = {cl68_2d:.2f})',
                  f'95 % ($\\Delta$NLL = {cl95_2d:.2f})']

for ax, (n1, n2) in zip(axes, pairs):
    g1 = grids_2d[n1]
    g2 = grids_2d[n2]
    Z  = delta_nll_2d[(n1, n2)]

    # Filled background showing the DNLL surface
    ax.contourf(g1, g2, Z, levels=30, cmap='Blues_r', alpha=0.7)

    # Confidence-region contours; build proxy artists for the legend
    proxy_handles = []
    for lvl, col, lbl in zip(contour_levels, contour_colors, contour_labels):
        ax.contour(g1, g2, Z, levels=[lvl], colors=[col], linewidths=2)
        proxy_handles.append(plt.Line2D([], [], color=col, lw=2, label=lbl))

    # MLE and true parameter values
    proxy_handles.append(
        ax.plot(mle_display[n1], mle_display[n2], 'g*', ms=12,
                label=f'MLE ({mle_display[n1]:.3f}, {mle_display[n2]:.3f})')[0]
    )
    proxy_handles.append(
        ax.plot(true_display[n1], true_display[n2], 'k+', ms=12, mew=2,
                label=f'True ({true_display[n1]:.3f}, {true_display[n2]:.3f})')[0]
    )

    ax.set_xlabel(labels[n1], fontsize=13)
    ax.set_ylabel(labels[n2], fontsize=13)
    ax.set_title(f'{labels[n1]} vs {labels[n2]}', fontsize=12)
    ax.legend(handles=proxy_handles, fontsize=7.5)

plt.suptitle(r'2-D Profile $\Delta$NLL and Confidence Regions', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Summary

**1-D confidence intervals** were extracted from the profile likelihood scan for each
parameter individually, profiling over the remaining three.

**2-D confidence regions** were extracted by scanning pairs of parameters simultaneously
and profiling over the other two.

**Key observations**
- The 1-D profile $\Delta$ NLL curves are approximately parabolic near the minimum, consistent with the Gaussian (quadratic-likelihood) approximation.
- The 2-D confidence regions are approximately elliptical but reveal correlations between parameters (e.g. a positive correlation between $\theta$ and $\sigma$).
- All true parameter values fall inside the 95 % confidence region, as expected for a well-specified model and sufficient statistics.

## Exercises

1. **Coverage check.** Generate 200 independent datasets (same true parameters, different seeds). For each dataset compute the 95 % profile-likelihood CI for $\theta$. What fraction of CIs contain the true value? Does it agree with the nominal 95 %?

2. **Parabolic (Hessian) approximation.** Estimate the uncertainty on $\hat{\theta}$ from the inverse Hessian of the NLL evaluated at the MLE (use `scipy.optimize.minimize` with `method='BFGS'` and inspect `result.hess_inv`). Compare the resulting symmetric CI with the profile-likelihood CI from section 3. When do they agree? When do they differ?

3. **Wilks' theorem validity.** Repeat the coverage study from exercise 1 for small datasets ($n = 100$). Does Wilks' theorem still hold? Plot the distribution of $2\,\Delta\text{NLL}$ at the true $\theta$ and compare it to the $\chi^2_1$ distribution.

4. **Effect of a systematic uncertainty.** Suppose the exponential rate $\lambda$ is not a free parameter but is constrained by an auxiliary measurement: $\lambda_{\rm aux} \sim \mathcal{N}(\lambda, \delta_\lambda^2)$ with $\delta_\lambda = 0.05$. Add this Gaussian constraint to the NLL and study how the CI for $\theta$ widens as $\delta_\lambda$ increases from 0 to 0.2.

5. **Comparing 1-D and 2-D intervals.** From the 2-D confidence region for $(\theta, \mu)$, obtain the marginal 1-D interval for $\theta$ by projecting the 2-D region onto the $\theta$ axis. Compare with the profile-likelihood 1-D interval. Why do they differ?